In [1]:
import os

RAW_VIDEO_DIR = "DATA/raw_videos"

video_paths = []
for actor in os.listdir(RAW_VIDEO_DIR):
    actor_dir = os.path.join(RAW_VIDEO_DIR, actor)
    if not os.path.isdir(actor_dir):
        continue

    for f in os.listdir(actor_dir):
        if f.endswith(".mp4"):
            video_paths.append(os.path.join(actor_dir, f))

print("Total videos found:", len(video_paths))


Total videos found: 1440


In [2]:
import cv2
import numpy as np

FRAME_DIR = "DATA/frames"
os.makedirs(FRAME_DIR, exist_ok=True)

def extract_5_frames(video_path, out_dir, size=(224,224)):
    os.makedirs(out_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return False

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 5:
        cap.release()
        return False

    idxs = np.linspace(0, total - 1, 5, dtype=int)
    idx_set = set(idxs.tolist())

    saved = 0
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id in idx_set:
            frame = cv2.resize(frame, size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Reject black / corrupt frames
            if frame.mean() < 5:
                cap.release()
                return False

            cv2.imwrite(
                os.path.join(out_dir, f"f{saved}.jpg"),
                cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            )
            saved += 1

        frame_id += 1

    cap.release()
    return saved == 5


In [3]:
failed = []

for vp in video_paths:
    base = os.path.splitext(os.path.basename(vp))[0]
    out_dir = os.path.join(FRAME_DIR, base)

    ok = extract_5_frames(vp, out_dir)
    if not ok:
        failed.append(base)

print("Failed videos:", len(failed))


Failed videos: 480


In [4]:
import pandas as pd

emotion_map = {
    "01": 0, "02": 1, "03": 2, "04": 3,
    "05": 4, "06": 5, "07": 6, "08": 7
}

records = []

for vp in video_paths:
    name = os.path.basename(vp).replace(".mp4", "")
    if name in failed:
        continue

    parts = name.split("-")
    if len(parts) != 7:
        continue

    frame_dir = os.path.join(FRAME_DIR, name)
    if not os.path.exists(frame_dir):
        continue

    records.append({
        "frames": frame_dir,
        "emotion": emotion_map[parts[2]],
        "actor": parts[-1]
    })

df = pd.DataFrame(records)
print("Clean samples:", len(df))
print(df.head())


Clean samples: 960
                             frames  emotion actor
0  DATA/frames\01-01-01-01-01-01-02        0    02
1  DATA/frames\01-01-01-01-01-02-02        0    02
2  DATA/frames\01-01-01-01-02-01-02        0    02
3  DATA/frames\01-01-01-01-02-02-02        0    02
4  DATA/frames\01-01-02-01-01-01-02        1    02


In [5]:
from sklearn.model_selection import train_test_split

actors = df["actor"].unique()
train_actors, val_actors = train_test_split(
    actors, test_size=0.2, random_state=42
)

train_df = df[df["actor"].isin(train_actors)].reset_index(drop=True)
val_df   = df[df["actor"].isin(val_actors)].reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df))


Train: 720 Val: 240


In [6]:
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

image_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class VisionTemporalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        imgs = []
        for i in range(5):
            img = Image.open(
                os.path.join(row["frames"], f"f{i}.jpg")
            ).convert("RGB")
            imgs.append(self.transform(img))

        imgs = torch.stack(imgs)   # [5, 3, 224, 224]
        label = torch.tensor(row["emotion"], dtype=torch.long)
        return imgs, label


In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    VisionTemporalDataset(train_df, image_transform),
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    VisionTemporalDataset(val_df, image_transform),
    batch_size=16,
    shuffle=False,
    num_workers=0
)


In [8]:
import torch.nn as nn
from torchvision import models

class CNN_LSTM(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        base = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1
        )
        self.cnn = nn.Sequential(*list(base.children())[:-1])

        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=256,
            batch_first=True,
            dropout=0.5
        )
    
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)

        feats = self.cnn(x).squeeze(-1).squeeze(-1)
        feats = feats.view(B, T, 512)

        _, (h, _) = self.lstm(feats)
        return self.fc(h[-1])


In [9]:
import torch.optim as optim
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_LSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)


E:\Anaconda\envs\gpu_env\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


In [10]:
def run_epoch(loader, train=True):
    model.train(train)
    correct = total = loss_sum = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if train:
            optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        if train:
            loss.backward()
            optimizer.step()

        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return loss_sum / len(loader), correct / total


In [11]:
for epoch in range(15):
    tl, ta = run_epoch(train_loader, True)
    vl, va = run_epoch(val_loader, False)
    print(f"Epoch {epoch+1} | Loss {tl:.4f} | Train {ta:.3f} | Val {va:.3f}")


Epoch 1 | Loss 1.7071 | Train 0.371 | Val 0.283
Epoch 2 | Loss 1.0591 | Train 0.615 | Val 0.371
Epoch 3 | Loss 0.6124 | Train 0.824 | Val 0.408
Epoch 4 | Loss 0.4484 | Train 0.856 | Val 0.533
Epoch 5 | Loss 0.3721 | Train 0.887 | Val 0.517
Epoch 6 | Loss 0.2532 | Train 0.932 | Val 0.537
Epoch 7 | Loss 0.2614 | Train 0.922 | Val 0.546
Epoch 8 | Loss 0.1658 | Train 0.960 | Val 0.467
Epoch 9 | Loss 0.0849 | Train 0.981 | Val 0.463
Epoch 10 | Loss 0.1518 | Train 0.958 | Val 0.425
Epoch 11 | Loss 0.1429 | Train 0.960 | Val 0.521
Epoch 12 | Loss 0.0814 | Train 0.976 | Val 0.492
Epoch 13 | Loss 0.1534 | Train 0.958 | Val 0.400
Epoch 14 | Loss 0.0620 | Train 0.983 | Val 0.542
Epoch 15 | Loss 0.0551 | Train 0.983 | Val 0.483
